# Enrich Support Tickets with Doris AI Functions and a Local Model

**Module 2: Call Language Models from SQL with AI Functions** · [Lesson](course.md) · [Quiz](quiz02_doris_ai_functions.ipynb)

In this Lab you run a small open model on your computer, next to the course's Doris sandbox, and call it from SQL. You label 14 support tickets with `AI_CLASSIFY` and `AI_SENTIMENT`, store the answers once, check them with ordinary SQL, extract facts, and summarize groups of tickets with `AI_AGG`. After each statement the Lab reads the model server's own counters, so you see how many requests and tokens the statement cost.

Time: about 40 minutes. The first run downloads the model server image (about 300 MB, 1.2 GB once unpacked) and the model file (1.1 GB); if you have not done Module 1, also the Doris image (about 1.9 GB, 4.5 GB once unpacked).

Before you start: Docker is running with at least 6 GB of memory, and you opened this notebook from the course's virtual environment (see Get started in the [course README](../../README.md)). Module 1 is not needed: section 1 starts the sandbox, or reuses it if Module 1 left it running.

### Initialize the Lab

Run this cell once. It finds the course directory, loads the course's lab helpers, and shows the addresses that the Lab uses: Doris, the model server as this computer reaches it, and the model server as the Doris BE reaches it.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import AiFunctionsLab

lab = AiFunctionsLab(lab_dir=COURSE_ROOT)

## 1. Start the sandbox and the local model

The Doris sandbox is the container `doris-ai` of Module 1: one Frontend (FE) and one Backend (BE) of Apache Doris 4.1.3. The model server is a second container, `doris-ai-llm`, on the same Docker network, `doris-ai`. Every port listens only on `127.0.0.1`:

| Port | Purpose | Variable to change it |
| --- | --- | --- |
| 39030 | FE MySQL protocol: the SQL of this Lab | `DORIS_AI_QUERY_PORT` |
| 38030 | FE HTTP | `DORIS_AI_HTTP_PORT` |
| 38040 | BE HTTP | `DORIS_AI_BE_HTTP_PORT` |
| 38080 | The model server, for this notebook | `DORIS_AI_LLM_PORT` |

To use other ports, set the variables before you start Jupyter. If you use your own Doris 4.x instead of the sandbox, skip the next cell and read "Use your own Doris" in lesson section 2.3 first.

In [ ]:
lab.prepare_environment()

**Expected result**

Five progress steps. The platform, Docker, and port checks pass. If Module 1 left the sandbox running, a card says "Reusing the running container doris-ai"; otherwise the Lab pulls the Doris image and starts the container, which takes one to two minutes after the download. The last table lists one FE and one BE, both alive, with the version `doris-4.1.3-rc02-7126cf65d96`, which is how the 4.1.3 release identifies itself.

The model server is the server of [llama.cpp](https://github.com/ggml-org/llama.cpp), an open-source program that runs language models on ordinary CPUs and answers OpenAI's chat completions API. The model is Qwen2.5 1.5B Instruct, quantized to 4 bits, under the Apache 2.0 license. The next cell downloads the model file once into the volume `doris-ai-llm-models`, checks its SHA-256, and starts the server. If huggingface.co is slow or blocked where you are, set `DORIS_AI_LLM_MODEL_URL` to the same file on a mirror before you start Jupyter: the checksum rejects any other file.

In [ ]:
lab.start_model()

**Expected result**

Four progress steps. On the first run, the download prints "Downloaded … MB of 1117 MB" every five seconds, then `qwen2.5-1.5b-instruct-q4_k_m.gguf: OK`; later runs print "Already downloaded" and check the file again, which takes a few seconds. The server answers a few seconds after it starts ("The model server answered after … seconds"), and the table lists one model, `qwen2.5-1.5b-instruct`. In a terminal, `docker ps --filter name=doris-ai` now lists two containers.

An AI function does not run in this notebook: the Doris BE sends the request to the model. So the model's address has to work from inside the BE's container, and that is not the address that this notebook uses. The next cell asks the model server for its models from both places.

In [ ]:
lab.check_model_routes()

**Expected result**

Four rows. From this computer, `http://127.0.0.1:38080/v1/models` answers with `qwen2.5-1.5b-instruct`. From the BE, `http://doris-ai-llm:8080/v1/models` answers HTTP 200: Docker finds the container by its name on the `doris-ai` network. The two `127.0.0.1` addresses give no answer from the BE (curl exit code 7: it could not connect), because inside the container `127.0.0.1` is the container itself. A green card names the address that the Resource will use.

## 2. Load the support tickets

The tickets are 14 short messages from customers, seven in English and seven in Chinese, in `datasets/support_tickets.csv`. A person gave each ticket a topic and a sentiment, in the columns `expected_topic` and `expected_sentiment`. The model never sees these two columns; section 6 compares its answers with them. The table goes into the course database `ai_course`, next to Module 1's `events`.

In [ ]:
lab.load_tickets()

**Expected result**

The `CREATE TABLE` statement, the first rows of the `INSERT`, and a card "Inserted 14 tickets." (on a rerun: "already holds the 14 course tickets; nothing to load"). The comparison table shows 14 tickets, 1001 to 1014, 7 in English and 7 in Chinese, 5 topics and 4 sentiments, each the same as in `datasets/support_tickets.yaml`.

In [ ]:
lab.sql("""
SELECT expected_topic, language, COUNT(*) AS tickets,
       ROUND(AVG(CHAR_LENGTH(raw_text)), 1) AS average_characters
FROM ai_course.support_tickets
GROUP BY expected_topic, language
ORDER BY expected_topic, language
""", title="The tickets by topic and language");

**Expected result**

Seven rows. `shipping` has the most tickets: three in Chinese and two in English. The Chinese tickets average 21.5 to 35 characters and the English ones 64 to 102.5. This is ordinary SQL: the model has done no work yet.

## 3. Create the AI Resource and make one call

An AI Resource is a Doris object that names a model service: the provider type, the endpoint, the model, and the settings of every request. AI functions take the Resource's name as their first argument. The provider type `local` is for a server that you run yourself and that needs no key; the endpoint is the address that the BE reaches, from section 1. The settings ask for the model's most likely answer (`ai.temperature` 0), stop each answer at 256 tokens (`ai.max_token`), and retry a failed request once, after one second.

In [ ]:
lab.create_resource()

**Expected result**

The `CREATE RESOURCE` statement, then the properties that Doris stored; Doris shows the provider type in capitals, `LOCAL`. The card points out that Doris did not contact the model: for the `local` provider, `CREATE RESOURCE` checks only that the required properties are there, so a wrong endpoint shows up at the first call.

In [ ]:
lab.ai_sql(
    "SELECT AI_GENERATE('ai_course_llm', 'Reply with the word READY.') AS reply",
    title="One call through Doris to the model",
);

**Expected result**

One row with a short reply, such as `READY.`; the wording can differ, because the answer is generated. The table of model work shows 1 request, about 140 prompt tokens, and a few generated tokens, in less than a second. The prompt is much longer than your six words because Doris sends its own instructions for `AI_GENERATE` with them (lesson section 2.2).

## 4. See how Doris batches rows

To save requests, Doris puts several rows into one request: a JSON array with one item per row, and the instruction to answer with a JSON array of as many strings, in the same order. The session variable `ai_context_window_size` caps the size of that array in bytes; its default, 131072, fits all 14 tickets. If the answer is not an array of exactly as many strings, Doris fails the whole statement.

The next cell asks for the sentiment of three tickets with the default batching. Small models often do not follow the array instruction, so it can fail; either outcome teaches something.

In [ ]:
THREE_SENTIMENTS = """
SELECT ticket_id, raw_text, AI_SENTIMENT('ai_course_llm', raw_text) AS sentiment
FROM ai_course.support_tickets
WHERE ticket_id IN (1001, 1007, 1010)
ORDER BY ticket_id
"""
lab.ai_sql(THREE_SENTIMENTS, title="Three tickets in one request");

**Expected result**

The model work shows 1 request for the three tickets. With this model the usual outcome is a card "Doris could not use the model's answer", with a message such as `Failed to parse ai_sentiment batch result, expected 3 items but got 1` or `Invalid batch result format, array element 0 is not a string`: the model answered, but not with an array of three strings, so no ticket got a sentiment. If you see three sentiments instead, this answer happened to have the right shape; the next cell still applies, because the same request can fail on the next run.

In [ ]:
lab.set_session(ai_context_window_size=1)
lab.ai_sql(THREE_SENTIMENTS, title="Three tickets, one request each");

**Expected result**

`SET ai_context_window_size = 1` succeeds. Every ticket is now larger than the window, so Doris sends each one on its own: the model work shows 3 requests, and the three rows have sentiments, usually `negative` for 1001, `positive` for 1007, and `mixed` for 1010. Each request repeats Doris's instructions, so the three requests carry more prompt tokens than the single one did. The Lab keeps this setting for the rest of the notebook, and sets it again if it has to reconnect.

## 5. Label every ticket once and keep the answers

A model call is slow, it costs something every time, and its answer can change from one run to the next. So call the model once per ticket and store what it answered, together with what produced the answer: the model, the SHA-256 of the file that holds it, and a version for the label list. The table `ticket_enrichment` keeps the raw answers; section 6 decides which of them can be used.

In [ ]:
lab.sql("""
CREATE TABLE IF NOT EXISTS ai_course.ticket_enrichment (
    ticket_id INT NOT NULL COMMENT "The ticket in support_tickets",
    topic_raw STRING NULL COMMENT "The answer of AI_CLASSIFY, as the model returned it",
    sentiment_raw STRING NULL COMMENT "The answer of AI_SENTIMENT, as the model returned it",
    model_name VARCHAR(64) NOT NULL COMMENT "The model that answered",
    model_sha256 CHAR(64) NOT NULL COMMENT "The SHA-256 of the model file",
    prompt_version VARCHAR(32) NOT NULL COMMENT "The version of the label list and functions",
    enriched_at DATETIME NOT NULL COMMENT "When the answers were stored"
)
UNIQUE KEY(ticket_id)
DISTRIBUTED BY HASH(ticket_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""", title="Create the table for the model's answers");

**Expected result**

The statement succeeds. `IF NOT EXISTS` keeps the table and its answers when you rerun the Lab.

In [ ]:
PENDING = """
SELECT t.ticket_id
FROM ai_course.support_tickets AS t
LEFT ANTI JOIN ai_course.ticket_enrichment AS e ON e.ticket_id = t.ticket_id
ORDER BY t.ticket_id
"""
# {ticket_id}, {model_name}, and {model_sha256} are filled in by lab.ai_sql.
ENRICH = """
INSERT INTO ai_course.ticket_enrichment
    (ticket_id, topic_raw, sentiment_raw, model_name, model_sha256, prompt_version, enriched_at)
SELECT t.ticket_id,
       AI_CLASSIFY('ai_course_llm', t.raw_text,
                   ['refund', 'shipping', 'quality', 'account', 'product_question']),
       AI_SENTIMENT('ai_course_llm', t.raw_text),
       '{model_name}', '{model_sha256}', 'labels-v1', NOW()
FROM ai_course.support_tickets AS t
LEFT ANTI JOIN ai_course.ticket_enrichment AS e ON e.ticket_id = t.ticket_id
WHERE t.ticket_id = {ticket_id}
"""
pending = lab.sql(PENDING, title="Tickets without stored answers")["ticket_id"]
lab.ai_sql(ENRICH, title="Store the model's answers for one ticket", each=pending,
           model_name=lab.model_name, model_sha256=lab.model_sha256);

**Expected result**

The first table lists the 14 tickets. The statement then runs once for each ticket, and the outcome table shows `wrote 1 row(s)` for every one. The model work shows 28 requests, two for each of 14 tickets: every AI function in a statement sends its own requests. It takes 10 to 15 seconds. If Doris could not use the answer for a ticket, its row says `failed` with Doris's message and the other tickets keep their answers; rerun the cell to try that ticket again.

In [ ]:
pending = lab.sql(PENDING, title="Tickets without stored answers")["ticket_id"]
lab.ai_sql(ENRICH, title="Store the model's answers for one ticket", each=pending,
           model_name=lab.model_name, model_sha256=lab.model_sha256);

**Expected result**

"No rows returned" for the tickets without stored answers, and a card "Nothing to do": the anti-join finds no ticket to send, so the statement does not run and the model does no work. Rerunning the Lab from the top keeps the stored answers too; only a ticket that failed above would be sent again.

## 6. Check the labels before you use them

`AI_CLASSIFY` asks the model to choose one of your labels, but Doris does not check that the answer is one of them; `AI_SENTIMENT` is in the same position with its four labels. The checks below are ordinary SQL on the stored answers: they cost no model work and give the same result every time you run them.

In [ ]:
lab.sql("""
SELECT e.ticket_id, t.language, e.topic_raw,
       e.topic_raw IN ('refund', 'shipping', 'quality', 'account', 'product_question') AS topic_is_a_label,
       e.sentiment_raw,
       e.sentiment_raw IN ('positive', 'negative', 'neutral', 'mixed') AS sentiment_is_a_label
FROM ai_course.ticket_enrichment AS e
JOIN ai_course.support_tickets AS t ON t.ticket_id = e.ticket_id
ORDER BY e.ticket_id
""", title="Is each answer one of the allowed labels?");

**Expected result**

Fourteen rows; in the two check columns, 1 means yes and 0 means no. The sentiments are usually all labels. Some topics usually are not exactly a label: they look like `[\"shipping\"]`, a label inside brackets and escaped quotes. The model answered in the array format that Doris asks for, but escaped the quotes, so Doris could not read the answer as an array and kept its whole text. How many answers look like this varies between runs and computers: 3 to 5 of the 14 in the runs that this Lab was checked with. Such an answer is not a topic yet; the exercise in section 10 recovers what can be recovered.

In [ ]:
lab.sql("""
SELECT t.ticket_id, t.raw_text, t.expected_topic, e.topic_raw, t.expected_sentiment, e.sentiment_raw
FROM ai_course.ticket_enrichment AS e
JOIN ai_course.support_tickets AS t ON t.ticket_id = e.ticket_id
WHERE e.topic_raw IN ('refund', 'shipping', 'quality', 'account', 'product_question')
  AND (e.topic_raw <> t.expected_topic OR e.sentiment_raw <> t.expected_sentiment)
ORDER BY t.ticket_id
""", title="Valid answers that differ from the labels a person chose");

**Expected result**

A few rows: tickets where a valid answer differs from the person's label. In the run that this Lab was checked with there were two, 1002 and 1013, and the model chose `product_question` for both. Read the text of each one before you call it a mistake: 1002 reports a broken product and asks for a refund, so `quality` and `refund` both fit, and 1013 asks a question about a delivery. Agreement on 14 tickets is not an accuracy figure; it shows which kinds of tickets need clearer labels, or a person.

In [ ]:
lab.ai_sql("""
SELECT topic_raw AS topic, sentiment_raw AS sentiment, COUNT(*) AS tickets
FROM ai_course.ticket_enrichment
WHERE topic_raw IN ('refund', 'shipping', 'quality', 'account', 'product_question')
  AND sentiment_raw IN ('positive', 'negative', 'neutral', 'mixed')
GROUP BY topic_raw, sentiment_raw
ORDER BY tickets DESC, topic, sentiment
""", title="A report on the stored, valid answers");

**Expected result**

About nine rows that count only the answers that passed the check; no combination has more than a few tickets. The model work shows 0 requests: a report on stored answers is ordinary SQL, as fast and as repeatable as any other query.

## 7. Extract facts and summarize

`AI_EXTRACT` returns one string per ticket, formatted as `label=value, label=value`, with an empty value for a label that the model did not find. Treat the values as candidates: an extracted order number is useful only if it really is in the ticket. The query below extracts three facts and checks the order number in the same statement, with `REGEXP_EXTRACT` and `LOCATE`. Three tickets show the cases: 1002 names an order, 1011 says the customer cannot find the order number, and 1012 names a person, an email address, and an order. The statement runs once per ticket, so that an answer that Doris cannot parse affects only its own ticket.

In [ ]:
lab.ai_sql("""
SELECT ticket_id, facts, order_id,
       CASE WHEN order_id = '' THEN 'no order number found'
            WHEN LOCATE(order_id, raw_text) > 0 THEN 'in the ticket'
            ELSE 'not in the ticket: do not use it'
       END AS order_id_check
FROM (
    SELECT ticket_id, raw_text, facts,
           REGEXP_EXTRACT(facts, 'order_id=([A-Za-z0-9-]*)', 1) AS order_id
    FROM (
        SELECT ticket_id, raw_text,
               AI_EXTRACT('ai_course_llm', raw_text, ['order_id', 'product', 'request']) AS facts
        FROM ai_course.support_tickets
        WHERE ticket_id = {ticket_id}
    ) AS answers
) AS parsed
""", title="Extract three facts from one ticket and check the order number", each=[1002, 1011, 1012]);

**Expected result**

Three requests, one per ticket: Doris computes `AI_EXTRACT` once per row even though the outer queries use its answer twice. For 1002 the answer contains `order_id=A-3017`, which is in the ticket. For 1011 the order number is empty, as it should be. For 1012, `A-2048` is in the ticket. The format around the values varies: the model sometimes wraps its answer in Markdown fences or puts each label on its own line, which is why the check reads only the `order_id=` part. A value that is "not in the ticket" was made up by the model. With this model, an answer for one ticket sometimes has a shape that Doris rejects (`expected 1 items but got 3`); that ticket's row in the outcome table says `failed`, and a rerun usually succeeds.

In [ ]:
lab.ai_sql("""
SELECT ticket_id, raw_text, AI_SUMMARIZE('ai_course_llm', raw_text) AS summary
FROM ai_course.support_tickets
WHERE ticket_id = {ticket_id}
""", title="Summarize one ticket", each=[1002, 1010]);

**Expected result**

Two requests and two summaries, each in the language of its ticket: Chinese for 1002, English for 1010. Read them against the text: a good summary of 1010 keeps both the praise for the camera and the complaint about the battery. A summary is for reading, not for counting, and it does not replace `raw_text` as the record of what the customer wrote.

## 8. Summarize groups with `AI_AGG`

`AI_AGG` is an aggregate function: it sends the rows of each group to the model together, with your instruction, and returns one text per group. How many rows a group has decides how much text goes into one request. In Doris 4.1.3, `AI_AGG` packs up to 128 KiB of text into a request whatever `ai_context_window_size` says, and summarizes a larger group in several steps.

In [ ]:
lab.ai_sql("""
SELECT expected_topic, COUNT(*) AS tickets,
       AI_AGG('ai_course_llm', raw_text,
              'Summarize the shared customer problem in one short sentence. Do not add facts.') AS group_summary
FROM ai_course.support_tickets
WHERE expected_topic IN ('shipping', 'quality')
GROUP BY expected_topic
ORDER BY expected_topic
""", title="One summary for each of two topics");

**Expected result**

Two rows, `quality` with 3 tickets and `shipping` with 5, and 2 requests, one per group. Each summary is about one sentence. Doris tells the model to answer in the language of the entries, so a group of Chinese and English tickets can come back in Chinese even though the instruction is in English. Check what a summary leaves out: the `quality` group has a happy customer (1007) next to two complaints. In the run that this Lab was checked with, both summaries were in Chinese, and the `quality` summary named only the headphones that do not charge.

## 9. Optional: try the other AI functions

Doris 4.1.3 has eleven AI functions for text, plus `EMBED` for vectors, which this module does not use. The Lab has used six of them. To try the other five on one ticket each, set `TRY_THE_OTHER_FUNCTIONS = True` and run the cell; it makes about five requests. With this small model, expect some of them to fail: `AI_FILTER` and `AI_SIMILARITY` need an answer in quotes, and the model tends to answer with a bare number. The table shows each function's result, or Doris's message.

In [ ]:
TRY_THE_OTHER_FUNCTIONS = False

OTHER_FUNCTIONS = {
    "AI_FILTER": "SELECT AI_FILTER('ai_course_llm', CONCAT('The customer asks for a refund: ', raw_text)) "
                 "FROM ai_course.support_tickets WHERE ticket_id = 1005",
    "AI_SIMILARITY": "SELECT AI_SIMILARITY('ai_course_llm', 'Where is my parcel?', raw_text) "
                     "FROM ai_course.support_tickets WHERE ticket_id = 1006",
    "AI_TRANSLATE": "SELECT AI_TRANSLATE('ai_course_llm', raw_text, 'English') "
                    "FROM ai_course.support_tickets WHERE ticket_id = 1005",
    "AI_FIXGRAMMAR": "SELECT AI_FIXGRAMMAR('ai_course_llm', 'my package have not arrive yet')",
    "AI_MASK": "SELECT AI_MASK('ai_course_llm', raw_text, ['name', 'email', 'order_id']) "
               "FROM ai_course.support_tickets WHERE ticket_id = 1012",
}
if TRY_THE_OTHER_FUNCTIONS:
    lab.try_functions(OTHER_FUNCTIONS)
else:
    print("Skipped. Set TRY_THE_OTHER_FUNCTIONS = True and run this cell again; it makes about five requests.")

**Expected result**

With `TRY_THE_OTHER_FUNCTIONS = False`, the cell prints that it skipped. With `True`, a table with one row per function, and about five requests. Typical results with this model: `AI_FILTER` and `AI_SIMILARITY` fail with `Invalid batch result format, array element 0 is not a string`, because the model answered `[0]` or a score without quotes; `AI_FIXGRAMMAR` corrects the sentence; `AI_TRANSLATE` may return its own input, wrapped in JSON, instead of a plain translation; `AI_MASK` replaces some of the name, the email address, and the order number with `[MASKED]`, but not always all three; in the run that this Lab was checked with, it left the order number `A-2048` in clear. None of these answers is safe to act on without a check.

## 10. Exercise: recover labels without calling the model

Section 6 found topic answers that are not exactly a label, such as `[\"shipping\"]`. Write one query on `ai_course.ticket_enrichment`, with no AI function, that returns each ticket with:

- `topic`: the answer itself when it is one of the five labels; otherwise the label inside the answer, when the answer has no other letters than that one label (only brackets, quotes, and other punctuation around it); otherwise NULL;
- `needs_a_person`: true when `topic` is NULL.

Hint: `REGEXP_EXTRACT(text, pattern, 1)` returns the first group of the pattern, or an empty string when the text does not match. The pattern `[^a-z_]*` matches characters that are not lowercase letters or underscores.

In [ ]:
# Your code here

**Reference solution.** Expand the next cell after you try. It reads the stored answers only, so it changes nothing and makes no request.

In [ ]:
lab.ai_sql("""
SELECT ticket_id, topic_raw, topic, topic IS NULL AS needs_a_person
FROM (
    SELECT ticket_id, topic_raw,
           CASE
               WHEN topic_raw IN ('refund', 'shipping', 'quality', 'account', 'product_question') THEN topic_raw
               ELSE NULLIF(REGEXP_EXTRACT(topic_raw,
                   '^[^a-z_]*(refund|shipping|quality|account|product_question)[^a-z_]*$', 1), '')
           END AS topic
    FROM ai_course.ticket_enrichment
) AS checked
ORDER BY ticket_id
""", title="Recovered topics");

**Expected result**

Fourteen rows. Every answer that was already a label keeps it; answers such as `[\"shipping\"]` become `shipping`; an answer with other words, or with two labels, stays NULL with `needs_a_person` true. The model work shows 0 requests. A rule like this is deterministic and can be reviewed, and because it reads stored answers, it gives the same result every time.

## 11. Completion criteria

- [ ] Section 1 shows the model server answering this computer and the BE, and the Resource uses the BE's address.
- [ ] Section 2 loaded 14 tickets that match `datasets/support_tickets.yaml`.
- [ ] `AI_GENERATE` answered through Doris with 1 request.
- [ ] You saw what `ai_context_window_size` changes: 1 request for three tickets, then 3 requests.
- [ ] `ticket_enrichment` holds answers for all 14 tickets, and storing them again made 0 requests.
- [ ] You checked the labels with SQL, and the report counted only valid answers, with 0 requests.
- [ ] You checked the extracted order numbers against the text of the tickets.
- [ ] Your exercise query recovers the same topics as the reference solution.

## 12. Stop or remove the environment

To free memory and keep everything for later, stop the model server and the sandbox. `lab.start()` and `lab.start_model()` start them again with the same data; the model file stays in its volume.

In [ ]:
lab.stop_model()
lab.stop()

In [ ]:
lab.start()
lab.start_model()

**Expected result**

`docker stop` prints each container's name, and two cards confirm the stops. The next cell starts the sandbox and waits until it is healthy, then checks the model file again ("Already downloaded" and `OK`) and starts the model server, in 20 to 60 seconds. The stored answers are still in `ai_course.ticket_enrichment`.

When you have finished the course, set `REMOVE_EVERYTHING = True` and run the next cell. It deletes both containers, `doris-ai` and `doris-ai-llm`, their three volumes (one holds the 1.1 GB model file), their network, and the files that Module 1 wrote. If you connected an MCP Host in Module 1, remove its `doris` entry first (Clean up, in the Module 1 lesson). With your own Doris, run `lab.drop_resource()` instead, and drop the two tables yourself.

In [ ]:
REMOVE_EVERYTHING = False

if REMOVE_EVERYTHING:
    lab.teardown()
else:
    print("Nothing removed. Set REMOVE_EVERYTHING = True and run this cell again.")

**Expected result**

With `REMOVE_EVERYTHING = False`, the cell prints that nothing was removed. With `True`, a card lists the removed containers, volumes, network, and files, and `docker ps -a --filter name=doris-ai` lists no containers.

## Next

Open [Quiz 2: Call Language Models from SQL with AI Functions](quiz02_doris_ai_functions.ipynb). This is the last module of the course for now; the [course README](../../README.md#roadmap) lists the modules that are planned.